# Notebook 02 — Payment Analysis

## Brazilian E-Commerce Public Dataset by Olist

### Objetivo executivo

Investigar **como os clientes pagam**, qual é a relação entre **meio de pagamento, ticket e parcelamento**, e quais comportamentos financeiros merecem atenção de gestores, investidores e acionistas.

### Perguntas de negócio

1. Quais meios de pagamento são mais utilizados?
2. Quais meios concentram maior valor financeiro?
3. Qual é a diferença entre um meio aparecer em um pedido e ser o meio principal do pedido?
4. O ticket médio muda de acordo com o meio de pagamento?
5. Como o parcelamento se distribui?
6. Compras de maior valor utilizam mais parcelas no cartão de crédito?
7. Qual é o valor financeiro associado a cada faixa de parcelas?
8. O comportamento de pagamento muda conforme a faixa de ticket?
9. Que conclusões podem ser levadas para a apresentação executiva?

> **Regra central de governança:** `payments` está na granularidade de pagamento, enquanto `orders_base` está na granularidade de pedido. Métricas de pedido devem ser calculadas depois da consolidação dos pagamentos por `order_id`.


## 1. Configuração e carregamento

Este notebook utiliza:

- `orders_base.csv` — uma linha por pedido, produzido no Notebook 01;
- `olist_order_payments_dataset.csv` — tabela bruta de pagamentos, preservando a granularidade original.

Os caminhos são relativos ao projeto e funcionam tanto a partir da raiz quanto da pasta `notebooks/`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

orders_base_path = PROCESSED_DIR / "orders_base.csv"
payments_path = RAW_DIR / "olist_order_payments_dataset.csv"

if not orders_base_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {orders_base_path}. "
        "Execute primeiro o Notebook 01."
    )

if not payments_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {payments_path}. "
        "Coloque os dados da Olist em data/raw/."
    )

orders_base = pd.read_csv(
    orders_base_path,
    parse_dates=["order_purchase_timestamp"]
)

payments = pd.read_csv(payments_path)

print("orders_base:", orders_base.shape)
print("payments:", payments.shape)


## 2. Inspeção inicial

In [ ]:
display(orders_base.head())
display(payments.head())


In [ ]:
payments.info()


In [ ]:
payment_nulls = (
    payments.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("nulls")
)

payment_nulls["null_pct"] = (
    payment_nulls["nulls"] / len(payments) * 100
)

payment_nulls


### Observação

Os principais campos utilizados neste notebook são:

- `order_id`: identifica o pedido;
- `payment_sequential`: sequência do pagamento dentro do pedido;
- `payment_type`: meio de pagamento;
- `payment_installments`: número de parcelas;
- `payment_value`: valor daquele registro de pagamento.

Um mesmo pedido pode possuir mais de um registro e mais de um tipo de pagamento.

Portanto:

- quantidade de registros em `payments` não representa quantidade de pedidos;
- `payment_value` representa o valor daquele registro;
- métricas de pedido exigem consolidação por `order_id`;
- análises por meio de pagamento devem distinguir **associação a um pedido** de **meio principal do pedido**.


## 3. Cobertura e consistência dos pagamentos

In [ ]:
payment_orders = payments["order_id"].nunique()

orders_with_payment = (
    orders_base["order_id"]
    .isin(payments["order_id"])
    .sum()
)

coverage = pd.Series({
    "orders_base": orders_base["order_id"].nunique(),
    "orders_with_payment": orders_with_payment,
    "payment_orders": payment_orders,
    "orders_without_payment":
        orders_base["order_id"].nunique() - orders_with_payment,
    "payment_rows": len(payments),
})

coverage


In [ ]:
payments_per_order = (
    payments.groupby("order_id")
    .size()
    .rename("payment_rows")
)

multiple_payment_orders = int(
    payments_per_order.gt(1).sum()
)

pd.Series({
    "orders_with_multiple_payment_records":
        multiple_payment_orders,
    "share_of_payment_orders_with_multiple_records":
        multiple_payment_orders / len(payments_per_order),
    "max_payment_records_per_order":
        payments_per_order.max(),
})


### Interpretação

A existência de múltiplos registros por pedido significa que:

- **quantidade de registros de pagamento não representa quantidade de pedidos**;
- um pedido pode usar mais de um meio de pagamento;
- participação de pedidos associados a um meio não é necessariamente exclusiva;
- para medidas financeiras no nível do pedido, pagamentos devem ser agregados antes do join com outras tabelas.


## 4. Consolidação por pedido e definição do meio principal

In [ ]:
# Valor total do pedido e características de pagamento
order_payment_total = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        payment_record_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
        installment_max=("payment_installments", "max"),
    )
)

# Valor de cada meio dentro do pedido
payment_type_value = (
    payments
    .groupby(
        ["order_id", "payment_type"],
        as_index=False
    )
    .agg(
        payment_value=("payment_value", "sum")
    )
)

# Meio principal = maior valor financeiro dentro do pedido
primary_idx = (
    payment_type_value
    .groupby("order_id")["payment_value"]
    .idxmax()
)

primary_payment = (
    payment_type_value
    .loc[
        primary_idx,
        ["order_id", "payment_type", "payment_value"]
    ]
    .rename(columns={
        "payment_type": "primary_payment_type",
        "payment_value": "primary_payment_value"
    })
    .reset_index(drop=True)
)

display(order_payment_total.head())
display(primary_payment.head())


### Duas leituras complementares

Neste notebook, serão usadas duas medidas diferentes:

1. **`order_association_share`**  
   Percentual de pedidos em que determinado meio aparece.  
   Como um pedido pode usar mais de um meio, essas participações **podem somar mais de 100%**.

2. **`primary_order_share`**  
   Percentual de pedidos em que determinado meio é o principal, definido pelo maior valor financeiro do pedido.  
   Como cada pedido possui apenas um meio principal, essas participações **somam 100%**.


## 5. Meio de pagamento — associação, dominância e valor

In [ ]:
payment_type_summary = (
    payments
    .groupby("payment_type")
    .agg(
        payment_records=("payment_type", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment_record=("payment_value", "mean"),
        median_payment_record=("payment_value", "median"),
    )
    .sort_values("payment_value", ascending=False)
)

payment_type_summary["share_records"] = (
    payment_type_summary["payment_records"]
    / payment_type_summary["payment_records"].sum()
)

payment_type_summary["order_association_share"] = (
    payment_type_summary["orders"]
    / payments["order_id"].nunique()
)

payment_type_summary["share_value"] = (
    payment_type_summary["payment_value"]
    / payment_type_summary["payment_value"].sum()
)

payment_type_summary


In [ ]:
primary_payment_summary = (
    primary_payment
    .groupby("primary_payment_type")
    .agg(
        orders=("order_id", "nunique")
    )
    .sort_values("orders", ascending=False)
)

primary_payment_summary["primary_order_share"] = (
    primary_payment_summary["orders"]
    / primary_payment_summary["orders"].sum()
)

primary_payment_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

plot_data = (
    payment_type_summary
    .sort_values("share_value", ascending=True)
)

ax.barh(
    plot_data.index,
    plot_data["share_value"] * 100
)

ax.set_title(
    "Participação do valor pago por meio de pagamento"
)
ax.set_xlabel("Participação financeira (%)")
ax.set_ylabel("Meio de pagamento")

plt.tight_layout()
plt.show()


### Insight executivo

A análise deve distinguir três dimensões:

- **popularidade/associação:** em quantos pedidos o meio aparece;
- **dominância:** em quantos pedidos ele é o meio principal;
- **valor:** quanto do valor financeiro total foi processado por esse meio.

Essa separação evita interpretar um meio usado parcialmente em um pedido como se ele tivesse sido o responsável exclusivo por toda a compra.


## 6. Ticket médio por meio de pagamento

Um pedido pode usar mais de um meio de pagamento. Por isso, utilizaremos duas leituras:

1. **Ticket dos pedidos associados ao meio** — o mesmo pedido pode aparecer em mais de uma categoria;
2. **Ticket por meio principal** — classificação exclusiva, uma categoria por pedido.

A segunda é mais adequada para comparações executivas.


In [ ]:
order_payment_types = (
    payments[
        ["order_id", "payment_type"]
    ]
    .drop_duplicates()
)

payment_order_level_association = (
    order_payment_types
    .merge(
        order_payment_total,
        on="order_id",
        how="left",
        validate="many_to_one"
    )
)

payment_ticket_by_association = (
    payment_order_level_association
    .groupby("payment_type")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("order_payment_value", "mean"),
        median_order_value=("order_payment_value", "median"),
    )
    .sort_values("avg_order_value", ascending=False)
)

payment_ticket_by_association


In [ ]:
payment_behavior = (
    order_payment_total
    .merge(
        primary_payment[
            ["order_id", "primary_payment_type"]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

payment_ticket_by_primary = (
    payment_behavior
    .groupby("primary_payment_type")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("order_payment_value", "mean"),
        median_order_value=("order_payment_value", "median"),
        total_order_value=("order_payment_value", "sum"),
    )
    .sort_values("avg_order_value", ascending=False)
)

payment_ticket_by_primary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

plot_data = (
    payment_ticket_by_primary
    .sort_values("avg_order_value")
)

ax.barh(
    plot_data.index,
    plot_data["avg_order_value"]
)

ax.set_title(
    "Ticket médio por meio principal do pedido"
)
ax.set_xlabel("Valor médio do pedido")
ax.set_ylabel("Meio principal")

plt.tight_layout()
plt.show()


## 7. Qualidade do campo de parcelamento

In [ ]:
installment_quality = (
    payments["payment_installments"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("payment_installments")
    .reset_index(name="payment_records")
)

installment_quality


In [ ]:
zero_installments = payments.loc[
    payments["payment_installments"].eq(0)
].copy()

pd.Series({
    "records_with_zero_installments":
        len(zero_installments),
    "orders_with_zero_installments":
        zero_installments["order_id"].nunique(),
    "payment_value_zero_installments":
        zero_installments["payment_value"].sum(),
})


### Regra de tratamento

Registros com `payment_installments = 0` são mantidos na tabela original, mas são excluídos das análises específicas de parcelamento, pois zero parcelas não representa uma condição econômica válida de parcelamento.

Essa decisão não altera os cálculos gerais de valor financeiro.


In [ ]:
payments_installment_valid = (
    payments.loc[
        payments["payment_installments"].ge(1)
    ]
    .copy()
)

print(
    "Registros válidos para análise de parcelamento:",
    len(payments_installment_valid)
)


## 8. Distribuição geral de parcelas

In [ ]:
installment_summary = (
    payments_installment_valid
    .groupby("payment_installments")
    .agg(
        payment_records=("payment_installments", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment=("payment_value", "mean"),
        median_payment=("payment_value", "median"),
    )
    .sort_index()
)

installment_summary["share_value"] = (
    installment_summary["payment_value"]
    / installment_summary["payment_value"].sum()
)

installment_summary.head(30)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = installment_summary.loc[
    installment_summary.index <= 15
]

ax.bar(
    plot_data.index.astype(str),
    plot_data["orders"]
)

ax.set_title(
    "Pedidos por quantidade de parcelas — principais faixas"
)
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Quantidade de pedidos")

plt.tight_layout()
plt.show()


### Observação

A visão geral do campo `payment_installments` é útil como diagnóstico. Entretanto, a interpretação econômica de parcelamento será concentrada no **cartão de crédito**, onde o número de parcelas possui significado mais direto.


## 9. Cartão de crédito × parcelamento

In [ ]:
credit_card = (
    payments.loc[
        payments["payment_type"].eq("credit_card")
        & payments["payment_installments"].ge(1)
    ]
    .copy()
)

credit_installments = (
    credit_card
    .groupby("payment_installments")
    .agg(
        payment_records=("payment_installments", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment=("payment_value", "mean"),
        median_payment=("payment_value", "median"),
    )
    .sort_index()
)

credit_installments["share_orders"] = (
    credit_installments["orders"]
    / credit_card["order_id"].nunique()
)

credit_installments["share_value"] = (
    credit_installments["payment_value"]
    / credit_card["payment_value"].sum()
)

credit_installments.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = credit_installments.loc[
    credit_installments.index <= 15
]

ax.bar(
    plot_data.index.astype(str),
    plot_data["share_value"] * 100
)

ax.set_title(
    "Cartão de crédito — participação do valor por parcelas"
)
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Participação financeira (%)")

plt.tight_layout()
plt.show()


## 10. Ticket × parcelas no cartão de crédito

In [ ]:
credit_order = (
    credit_card
    .groupby("order_id", as_index=False)
    .agg(
        credit_card_value=("payment_value", "sum"),
        credit_installments=("payment_installments", "max"),
        credit_payment_records=("payment_sequential", "count"),
    )
    .merge(
        order_payment_total[
            ["order_id", "order_payment_value"]
        ],
        on="order_id",
        how="left",
        validate="one_to_one"
    )
)

credit_order["credit_share_of_order"] = (
    credit_order["credit_card_value"]
    / credit_order["order_payment_value"]
)

credit_order.head()


In [ ]:
credit_installment_ticket = (
    credit_order
    .groupby("credit_installments")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("order_payment_value", "mean"),
        median_order_value=("order_payment_value", "median"),
        avg_credit_value=("credit_card_value", "mean"),
    )
    .sort_index()
)

credit_installment_ticket.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = credit_installment_ticket.loc[
    credit_installment_ticket.index <= 12
]

ax.plot(
    plot_data.index,
    plot_data["median_order_value"],
    marker="o"
)

ax.set_title(
    "Cartão de crédito — ticket mediano por número de parcelas"
)
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Ticket mediano do pedido")

plt.tight_layout()
plt.show()


### Teste quantitativo da hipótese

A correlação abaixo mede associação entre número de parcelas e valor do pedido **dentro do universo de pedidos com cartão de crédito**.

A correlação não prova causalidade.


In [ ]:
credit_spearman = credit_order[
    ["order_payment_value", "credit_installments"]
].corr(method="spearman").iloc[0, 1]

credit_pearson = credit_order[
    ["order_payment_value", "credit_installments"]
].corr(method="pearson").iloc[0, 1]

pd.Series({
    "credit_card_spearman":
        credit_spearman,
    "credit_card_pearson":
        credit_pearson,
})


### Como interpretar

- correlação positiva: pedidos de maior valor tendem a apresentar mais parcelas;
- correlação próxima de zero: pouca associação linear/monotônica;
- correlação negativa: pedidos de maior valor tendem a utilizar menos parcelas.

**Não interpretar correlação como causalidade.**


## 11. Distribuição do ticket por parcelas no cartão

In [ ]:
q99_credit = (
    credit_order["order_payment_value"]
    .quantile(0.99)
)

box_data = credit_order.loc[
    credit_order["order_payment_value"].le(q99_credit)
    & credit_order["credit_installments"].between(1, 12)
].copy()

fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=box_data,
    x="credit_installments",
    y="order_payment_value",
    ax=ax
)

ax.set_title(
    "Cartão de crédito — distribuição do ticket por parcelas "
    "(até P99 e 12 parcelas)"
)
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Valor do pedido")

plt.tight_layout()
plt.show()


## 12. Meio principal × faixa de ticket

Para transformar a análise em comportamento, os pedidos serão classificados por **meio principal**, garantindo que cada pedido apareça em apenas uma categoria.

Assim, cada linha da tabela de composição por faixa de ticket soma 100%.


In [ ]:
bins = [
    -np.inf,
    50,
    100,
    250,
    500,
    1000,
    np.inf
]

labels = [
    "Até R$ 50",
    "R$ 50–100",
    "R$ 100–250",
    "R$ 250–500",
    "R$ 500–1.000",
    "Acima de R$ 1.000",
]

payment_behavior["ticket_band"] = pd.cut(
    payment_behavior["order_payment_value"],
    bins=bins,
    labels=labels
)

ticket_payment_mix = (
    pd.crosstab(
        payment_behavior["ticket_band"],
        payment_behavior["primary_payment_type"],
        normalize="index"
    )
    * 100
)

ticket_payment_mix


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ticket_payment_mix.plot(
    kind="bar",
    stacked=True,
    ax=ax
)

ax.set_title(
    "Composição do meio principal por faixa de ticket"
)
ax.set_xlabel("Faixa de ticket")
ax.set_ylabel("Participação (%)")
ax.legend(
    title="Meio principal",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


### Insight executivo

Essa visualização permite responder:

> “O meio principal de pagamento muda conforme o valor da compra?”

Por exemplo, maior concentração de cartão de crédito nas faixas de maior ticket pode justificar investigações sobre parcelamento, limite de crédito e estratégias comerciais.

O resultado deve ser descrito como associação observada, não como relação causal.


## 13. Consolidação executiva

In [ ]:
executive_payment_summary = (
    payment_type_summary[
        [
            "orders",
            "payment_value",
            "order_association_share",
            "share_value",
            "avg_payment_record",
            "median_payment_record",
        ]
    ]
    .copy()
)

executive_payment_summary


In [ ]:
payment_kpis = pd.DataFrame({
    "kpi": [
        "total_payment_value",
        "payment_orders",
        "payment_records",
        "avg_order_payment_value",
        "median_order_payment_value",
        "orders_with_multiple_payment_records",
        "credit_card_primary_order_share",
        "credit_card_financial_share",
        "credit_card_spearman_ticket_installments",
    ],
    "value": [
        payments["payment_value"].sum(),
        payments["order_id"].nunique(),
        len(payments),
        order_payment_total["order_payment_value"].mean(),
        order_payment_total["order_payment_value"].median(),
        multiple_payment_orders,
        primary_payment_summary.loc[
            "credit_card",
            "primary_order_share"
        ] if "credit_card" in primary_payment_summary.index else np.nan,
        payment_type_summary.loc[
            "credit_card",
            "share_value"
        ] if "credit_card" in payment_type_summary.index else np.nan,
        credit_spearman,
    ],
})

payment_kpis


## 14. Regras para a narrativa executiva

Ao preparar o relatório e a apresentação, priorizar quatro mensagens:

### 1. Associação
Em quantos pedidos cada meio de pagamento aparece?

### 2. Dominância
Qual meio é o principal em cada pedido?

### 3. Valor
Qual meio concentra maior participação financeira?

### 4. Parcelamento
No cartão de crédito, existe associação entre maior ticket e maior número de parcelas?

### Estrutura narrativa sugerida

> **“O comportamento de pagamento é concentrado em [meio]. Entretanto, a participação financeira não é necessariamente igual à presença nos pedidos, indicando diferenças de ticket e composição. No cartão de crédito, o parcelamento concentra-se em [faixa] e apresenta [associação positiva/negativa/fraca] com o valor dos pedidos. Esse comportamento pode apoiar decisões sobre conversão, parcelamento e segmentação.”**

Substitua os campos entre colchetes pelos resultados efetivamente observados após a execução.


## 15. Sanity checks

In [ ]:
assert order_payment_total["order_id"].is_unique, (
    "order_payment_total deve ter uma linha por pedido."
)

assert primary_payment["order_id"].is_unique, (
    "primary_payment deve ter uma linha por pedido."
)

assert payment_behavior["order_id"].is_unique, (
    "payment_behavior deve ter uma linha por pedido."
)

assert payments_installment_valid[
    "payment_installments"
].ge(1).all(), (
    "Há parcelas inválidas na base de parcelamento."
)

assert np.isclose(
    payment_type_summary["share_value"].sum(),
    1.0
), "share_value deveria somar 100%."

assert np.isclose(
    primary_payment_summary[
        "primary_order_share"
    ].sum(),
    1.0
), "primary_order_share deveria somar 100%."

ticket_mix_sums = ticket_payment_mix.sum(axis=1)

assert np.allclose(
    ticket_mix_sums.dropna(),
    100.0
), "Cada faixa de ticket deveria somar 100%."

assert credit_order[
    "credit_installments"
].ge(1).all(), (
    "Há parcelamento inválido na base de cartão."
)

print("Sanity checks concluídos com sucesso.")


## 16. Exportação dos resultados

As tabelas abaixo podem ser reutilizadas no relatório, na apresentação executiva e no Power BI.


In [ ]:
payment_type_summary.to_csv(
    PROCESSED_DIR / "payment_type_summary.csv"
)

primary_payment_summary.to_csv(
    PROCESSED_DIR / "primary_payment_summary.csv"
)

installment_summary.to_csv(
    PROCESSED_DIR / "installment_summary.csv"
)

credit_installments.to_csv(
    PROCESSED_DIR / "credit_card_installment_summary.csv"
)

credit_installment_ticket.to_csv(
    PROCESSED_DIR / "credit_card_installment_ticket.csv"
)

ticket_payment_mix.to_csv(
    PROCESSED_DIR / "ticket_payment_mix.csv"
)

payment_behavior.to_csv(
    PROCESSED_DIR / "payment_behavior_order_level.csv",
    index=False
)

payment_kpis.to_csv(
    PROCESSED_DIR / "payment_executive_kpis.csv",
    index=False
)

print(
    "Resultados exportados para:",
    PROCESSED_DIR.resolve()
)


## 17. Checklist

- [x] Cobertura dos pagamentos verificada.
- [x] Granularidade de pagamento e pedido diferenciadas.
- [x] Associação a meios de pagamento calculada.
- [x] Meio principal por pedido definido.
- [x] Participação financeira calculada.
- [x] Ticket por meio principal analisado.
- [x] `payment_installments = 0` investigado e excluído somente das análises de parcelamento.
- [x] Distribuição geral de parcelas analisada.
- [x] Cartão de crédito analisado separadamente.
- [x] Ticket × parcelas analisado especificamente no cartão.
- [x] Correlações de Pearson e Spearman calculadas no universo de cartão.
- [x] Meio principal × faixa de ticket analisado.
- [x] Sanity checks executados.
- [x] Tabelas exportadas para reutilização.

### Próximo notebook

**Notebook 03 — Customer RFM**

A próxima etapa será transformar o histórico de pedidos em uma visão de cliente usando `customer_unique_id`, calculando **Recency, Frequency e Monetary**, criando segmentos e identificando quais grupos concentram o maior valor econômico.

> Observação: o Notebook 03 deverá utilizar `primary_payment_type` no cruzamento opcional com os outputs deste notebook.
